# Solving CyberGym Challenges with Sec-Gemini SDK

This notebook demonstrates how to use the Sec-Gemini Python SDK to interact with CyberGym challenges using specific MCPs.

In [ ]:
!pip install -q sec-gemini

## Authentication

Set your API key. Get one from [secgemini.google/keys](https://secgemini.google/keys).

In [ ]:
import os

# Option 1: From environment variable
API_KEY = os.environ.get("SEC_GEMINI_API_KEY")

# Option 2: From Colab secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        API_KEY = userdata.get("SEC_GEMINI_API_KEY")
    except (ImportError, Exception):
        pass

# Option 3: Paste directly (not recommended for shared notebooks)
if not API_KEY:
    API_KEY = "YOUR_API_KEY_HERE"

assert API_KEY and API_KEY != "YOUR_API_KEY_HERE", "Please set your API key"

## Connect and Setup Session

Connect to Sec-Gemini, create a session, and configure the CyberGym MCP.

In [ ]:
from sec_gemini import SecGemini

client = SecGemini(api_key=API_KEY)
await client.start()

session = await client.sessions.create()
print(f"Session: {session.id}")

# Set the CyberGym and Skills MCPs
await session.mcps.set(
    mcps=[
        "https://mcp-cybergym-171354917004.us-central1.run.app/mcp",
        "https://mcp-skills-171354917004.us-central1.run.app/mcp",
    ]
)
print("MCPs configured.")

## Solve a Challenge

Prompt the agent to solve a specific challenge. We pass the challenge name in metadata.

In [ ]:
prompt = "Solve the cybergym challenge. Use your cybergym tools to understand the task and find the flag."
challenge_name = "crypto-hash-cash"

print(f"Prompting agent for challenge: {challenge_name}")
await session.prompt(prompt, meta={"cybergym_challenge_name": challenge_name})

async for msg in session.messages.stream():
    print(f"\nMessage> Title: {msg.get('title')}")
    print(f"         Content: {msg.get('content')}")

## Cleanup

In [ ]:
await session.delete()
await client.close()
print("Done!")